# Constraint Analysis
### Roadrunner propulsion reverse-engineering study · notebook 10 of 14

**Purpose.** Repeat the constraint analysis of notebook 6 with the engine's own thrust lapse from
the off-design cycle analysis (notebook 9) instead of the algebraic one. It answers:

> *Which requirement sizes the engine when the lapse is the cycle's, and how far do the other
> requirements move?*

Nothing else changes: the requirements (ADR-0027), the mission, the hand-estimate polar and the
coupling through `solve_design_point()` are those of the preliminary analysis (ADR-0029). The
lapse is interpolated from the maximum-power grid of `out/off_design.yaml`; outside that grid the
analysis refuses to run rather than extrapolate (ADR-0033). The preliminary results are
recomputed alongside and drawn dashed.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import yaml

from roadrunner_re import analysis_outputs as ao
from roadrunner_re import reports
from roadrunner_re.analysis_inputs import load_analysis_inputs
from roadrunner_re.design_point import solve_design_point
from roadrunner_re.engine_model import CycleEngine
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots import analysis as plots

paths = nb_setup()
preliminary_inputs = load_analysis_inputs()                       # algebraic lapse, constant TSFC
off_design = yaml.safe_load((paths.out / "off_design.yaml").read_text(encoding="utf-8"))
inputs = preliminary_inputs.with_engine(CycleEngine(off_design))   # pyCycle lapse and part-throttle SFC
preliminary = solve_design_point(preliminary_inputs)               # the single call site (ADR-0029)
design = solve_design_point(inputs)
sweep = ao.takeoff_mass_sweep(inputs, ao.sweep_masses(inputs))

---
## 1 · What changed: the lapse

The installed thrust lapse at the conditions of the requirements, cycle against algebraic
(thrust over standard-day sea-level static thrust, mission hot day).

In [ ]:
print(f"{'requirement':16s}{'alt [m]':>8s}{'Mach':>6s}{'cycle':>8s}{'algebraic':>11s}")
for c, p in zip(design.constraints, preliminary.constraints):
    if c.lapse is not None:
        print(f"{c.id:16s}{c.altitude_m:8.0f}{c.mach:6.2f}{c.lapse:8.3f}{p.lapse:11.3f}")

---
## 2 · The requirements as thrust loading

Mattingly's master equation as in notebook 6, with $\alpha$ from the cycle. $\beta$ is the weight
fraction at the start of the requirement's segment from the *cycle-based* mission (the coupling
iterates, so it is not the preliminary $\beta$).

In [ ]:
comparison = ao.constraint_comparison(preliminary, design)
print(reports.design_constraint_table(design))
print()
print(reports.constraint_comparison_table(comparison))
print(f"\nconverged in {design.iterations} iterations")

---
## 3 · The constraint diagram

In [ ]:
fig = plots.constraint_diagram(inputs, design, paths.figures / "constraint_diagram_cycle.png", reference=(preliminary_inputs, preliminary))

Solid: cycle lapse. Dashed: preliminary. The landing floor lies far above the plotted range.

---
## 4 · Take-off mass sweep

In [ ]:
fig = plots.thrust_sweep_figure(sweep, paths.figures / "thrust_sweep_cycle.png")

---
## 5 · Handoff

In [ ]:
content = ao.constraints_out(inputs, design, sweep, notebook="constraint_analysis", comparison=comparison)
handoff = ao.write_handoff(content, paths.out / "constraints.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **The vertical launch still sizes the engine.** T_SL/W_TO is 1.469 against 1.474 preliminary (612 N
  per engine against 614 N): the launch lapse is 0.885 against 0.882 and the launch does not depend on
  flight lapse at all. The conclusion of notebook 6 survives the cycle.
- **Flight requirements get harder, not easier.** The cycle's lapse is lower than the algebraic one at
  every flight condition (dash 0.696 against 0.737, manoeuvre 0.647 against 0.704), so those
  requirements need 6 to 9 % more thrust loading. The manoeuvre is the closest competitor at 1.005, about
  68 % of the launch requirement; the margin to the binding constraint is large.
- **The weight fractions move too.** The cycle-based mission burns more fuel (notebook 11), so the
  start-of-segment weights of the later requirements differ from the preliminary ones.
- **The landing floor is an upper bound** (16.6 against 18.3, set by the catalogue idle fraction, which
  the cycle cannot derive); it is far above anything the launch demands.
- **Limits.** The lapse is the model's generic-map lapse (notebook 9) and is interpolated inside its
  grid only; thrust is uninstalled (ADR-0011).